# 01 · Model & load — the cross-server pipeline

The **data-modeling** MCP server designs an Airports/ROUTEs schema and generates the ingest Cypher; the **cypher** server runs it to load real OpenFlights data. No hand-written Cypher. Run `build.py` for the full ~67k-route load; this notebook shows the pipeline on a small sample.

In [1]:
import warnings; warnings.filterwarnings("ignore")   # quiet 3rd-party import warnings
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # examples/demos
from _common import clients, console
print("helpers ready — no API key needed (the MCP servers are pure tools)")

helpers ready — no API key needed (the MCP servers are pure tools)


## Design + generate (data-modeling server — no database)

In [2]:
def prop(n,t="STRING"): return {"name":n,"type":t,"source":None,"description":None}
AIRPORT={"label":"Airport","key_property":prop("iata"),
         "properties":[prop("name"),prop("city"),prop("country"),prop("lat","FLOAT"),prop("lon","FLOAT")]}
ROUTE={"type":"ROUTE","start_node_label":"Airport","end_node_label":"Airport",
       "key_property":prop("airline"),"properties":[prop("airline"),prop("equipment"),prop("stops","INTEGER")],"metadata":{}}
DM={"nodes":[AIRPORT],"relationships":[ROUTE]}

async def design():
    async with clients.data_modeling_client() as dm:
        print("validate_data_model:", clients.data(await dm.call_tool("validate_data_model",{"data_model":DM})))
        print("\nMermaid:\n", clients.text(await dm.call_tool("get_mermaid_config_str",{"data_model":DM})))
        print("\nNode ingest Cypher:\n", clients.text(await dm.call_tool("get_node_cypher_ingest_query",{"node":AIRPORT})))
        global NODE_Q, REL_Q
        NODE_Q=clients.text(await dm.call_tool("get_node_cypher_ingest_query",{"node":AIRPORT}))
        REL_Q=clients.text(await dm.call_tool("get_relationship_cypher_ingest_query",
              {"data_model":DM,"relationship_type":"ROUTE","relationship_start_node_label":"Airport","relationship_end_node_label":"Airport"}))
await design()

validate_data_model: True

Mermaid:
 graph TD
%% Nodes
Airport["Airport<br/>iata: STRING | KEY<br/>name: STRING<br/>city: STRING<br/>country: STRING<br/>lat: FLOAT<br/>lon: FLOAT"]

%% Relationships
Airport -->|ROUTE<br/>airline: STRING | KEY<br/>equipment: STRING<br/>stops: INTEGER| Airport


%% Styling 
classDef node_0_color fill:#e3f2fd,stroke:#1976d2,stroke-width:3px,color:#000,font-size:12px
class Airport node_0_color


        


Node ingest Cypher:
 UNWIND %(records)s as record
MERGE (n: "Airport" {iata: record.iata})
SET n += {name: record.name, city: record.city, country: record.country, lat: record.lat, lon: record.lon}


## Load a sample through the cypher write tool, then introspect

In [3]:
from _common import datautil
async def load_and_show():
    aps=datautil.airports(limit=400); valid={a["iata"] for a in aps}
    rts=datautil.routes(limit=1500, valid_iata=valid)
    recs=[{"sourceId":r["src"],"targetId":r["dst"],"airline":r["airline"],"equipment":r["equipment"],"stops":r["stops"]} for r in rts]
    async with clients.cypher_client("mcp_flights","flights") as cy:
        await cy.call_tool("write_agensgraph_cypher",{"query":NODE_Q,"params":{"records":aps}})
        await cy.call_tool("write_agensgraph_cypher",{"query":REL_Q,"params":{"records":recs}})
        schema=clients.data(await cy.call_tool("get_agensgraph_schema",{}))
        for label,info in schema.items():
            console.kv(label, f"{info.get('count')} nodes; rels {list((info.get('relationships') or {}))}")
await load_and_show()

  Airport                    6072 nodes; rels ['ROUTE']
